In [ ]:
import subprocess, sys

WHEEL_URL = (
    "https://github.com/JamePeng/llama-cpp-python/releases/download/"
    "v0.3.33-cu128-Basic-linux-20260315/"
    "llama_cpp_python-0.3.33+cu128.basic-cp312-cp312-linux_x86_64.whl"
)

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    WHEEL_URL, "--force-reinstall", "--no-deps", "--progress-bar=on",
], stdout=sys.stdout, stderr=sys.stderr)

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "gradio", "huggingface_hub", "nvidia-cublas-cu12", "nvidia-cuda-runtime-cu12",
    "--progress-bar=on",
], stdout=sys.stdout, stderr=sys.stderr)

print("\nAll packages installed.")

In [ ]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive",
    filename="Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q6_K.gguf",
)
print(f"Model downloaded to: {model_path}")

In [ ]:
import ctypes, os, site, glob

for sp in site.getsitepackages():
    for lib_dir in glob.glob(os.path.join(sp, "nvidia", "*", "lib")):
        for so in sorted(glob.glob(os.path.join(lib_dir, "*.so*"))):
            try:
                ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass

from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,
    n_ctx=8192,
    verbose=True,
)
print("Model loaded.")

In [ ]:
import gradio as gr


def build_messages(history, system_prompt):
    messages = [{"role": "system", "content": system_prompt}]
    for msg in history:
        messages.append({"role": msg["role"], "content": msg["content"]})
    return messages


def bot_respond(history, system_prompt, temperature, top_p, top_k, max_tokens, repeat_penalty):
    api_messages = build_messages(history, system_prompt)

    response = llm.create_chat_completion(
        messages=api_messages,
        stream=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        max_tokens=int(max_tokens),
        repeat_penalty=repeat_penalty,
    )

    history.append({"role": "assistant", "content": ""})
    for chunk in response:
        delta = chunk["choices"][0]["delta"]
        if "content" in delta:
            history[-1]["content"] += delta["content"]
            yield history


def user_submit(message, history):
    history.append({"role": "user", "content": message})
    return "", history


with gr.Blocks(title="Qwen 3.5 35B-A3B Uncensored") as demo:
    gr.Markdown("# Qwen 3.5 35B-A3B Uncensored Playground")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=600)
            msg = gr.Textbox(placeholder="Type a message…", show_label=False, container=False)
            with gr.Row():
                submit_btn = gr.Button("Send", variant="primary")
                clear_btn = gr.ClearButton([msg, chatbot], value="Clear")

        with gr.Column(scale=1):
            gr.Markdown("### Parameters")
            system_prompt = gr.Textbox(
                value="You are a helpful assistant.",
                label="System Prompt",
                lines=3,
            )
            temperature = gr.Slider(0.0, 2.0, value=1.0, step=0.05, label="Temperature")
            top_p = gr.Slider(0.0, 1.0, value=0.95, step=0.05, label="Top-p")
            top_k = gr.Slider(1, 100, value=20, step=1, label="Top-k")
            max_tokens = gr.Slider(64, 32768, value=8192, step=64, label="Max Tokens")
            repeat_penalty = gr.Slider(1.0, 2.0, value=1.0, step=0.05, label="Repeat Penalty")

    params = [system_prompt, temperature, top_p, top_k, max_tokens, repeat_penalty]

    msg.submit(user_submit, [msg, chatbot], [msg, chatbot], queue=False).then(
        bot_respond, [chatbot] + params, chatbot
    )
    submit_btn.click(user_submit, [msg, chatbot], [msg, chatbot], queue=False).then(
        bot_respond, [chatbot] + params, chatbot
    )

demo.queue()
demo.launch(share=True)